<!--
 Licensed to the Apache Software Foundation (ASF) under one
 or more contributor license agreements.  See the NOTICE file
 distributed with this work for additional information
 regarding copyright ownership.  The ASF licenses this file
 to you under the Apache License, Version 2.0 (the
 "License"); you may not use this file except in compliance
 with the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing,
 software distributed under the License is distributed on an
 "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
 KIND, either express or implied.  See the License for the
 specific language governing permissions and limitations
 under the License.
-->

# Buildings inside a flood extent — and which are critical infrastructure

An end-to-end disaster-response workflow. We answer:

> **A flood happened over this AOI. Which buildings are inside the flood extent, and how many of them are hospitals, schools, or fire stations?**

**Pipeline:**
1. Synthesize a Sentinel-1-VV-style backscatter raster (low values where water sits flat — that's the flood signature SAR sees).
2. Threshold with `RS_MapAlgebra` → binary water mask raster.
3. Vectorize the mask: `RS_PixelAsPolygons` + `explode` + filter + `ST_Union_Agg` → a single dissolved flood polygon.
4. `ST_Buffer` slightly to absorb SAR speckle.
5. Synthesize building footprints labelled with infrastructure type (`residential` / `hospital` / `school` / `fire_station`).
6. `ST_Intersects` spatial join to find affected buildings; flag any affected building whose `type IN ('hospital', 'school', 'fire_station')` as critical.
7. Write the affected-buildings table as **GeoParquet 1.1** (auto covering-bbox + projjson CRS), read back, plot.

All inputs are synthesized in the notebook so it runs offline. To replace the synthesis cell with real Sentinel-1 imagery in production, point Sedona's [STAC reader](../tutorial/files/stac-sedona-spark.md) at an `earth-search` collection and feed the chosen scene's VV-polarization asset URL into the same `sedona.read.format("raster")` call below.

**Requires Sedona ≥ 1.9.0** for the auto-tiling raster reader (GH-2672) and the GeoParquet 1.1 writer's auto-populated covering-bbox + projjson CRS metadata (GH-2646, GH-2664).

## 1. Connect to Spark through SedonaContext

In [ ]:
from sedona.spark import SedonaContext

config = SedonaContext.builder().master("spark://localhost:7077").getOrCreate()
sedona = SedonaContext.create(config)

## 2. Synthesize a Sentinel-1-VV-style backscatter raster

Sentinel-1 SAR over flat water returns very low backscatter (water specularly reflects radar away from the satellite); over land the surface scatters energy back. We simulate that contrast with a single-band `float32` raster: high values everywhere except a circular flood patch in the south-west quadrant which we drop to low values. Written as a tiled GeoTIFF so the Sedona raster reader accepts it.

In [ ]:
import os

import numpy as np
import rasterio
from rasterio.transform import from_bounds

WORK = "/tmp/flood-snapshot"
os.makedirs(WORK, exist_ok=True)

AOI = (-91.10, 41.50, -91.00, 41.60)
W = H = 128
transform = from_bounds(*AOI, W, H)
rng = np.random.default_rng(11)

ys, xs = np.mgrid[0:H, 0:W]
flood_mask = ((xs - 32) ** 2 + (ys - 96) ** 2) < 28**2  # circle in SW corner
backscatter = 0.5 + 0.05 * rng.standard_normal((H, W))  # land
backscatter = np.where(
    flood_mask, 0.05 + 0.02 * rng.standard_normal((H, W)), backscatter
)
backscatter = backscatter.clip(0, 1).astype("float32")

scene_path = f"{WORK}/sentinel1_vv.tif"
with rasterio.open(
    scene_path,
    "w",
    driver="GTiff",
    tiled=True,
    blockxsize=128,
    blockysize=128,
    height=H,
    width=W,
    count=1,
    dtype="float32",
    crs="EPSG:4326",
    transform=transform,
) as dst:
    dst.write(backscatter, 1)
    dst.set_band_description(1, "vv")

print(
    f"backscatter min={backscatter.min():.2f} max={backscatter.max():.2f} "
    f"flood pixels={int(flood_mask.sum())}/{H*W}"
)

## 3. Threshold the backscatter into a water-mask raster

Single-raster `RS_MapAlgebra` with a comparison expression: every pixel whose VV backscatter is below `0.2` becomes 1 (water), every other pixel becomes 0 (dry). The result is a `uint8` mask raster the same size and CRS as the input.

In [ ]:
scene = sedona.read.format("raster").load(scene_path)
scene.createOrReplaceTempView("scene")

mask = sedona.sql("""
    SELECT x, y,
           RS_MapAlgebra(
               rast, 'B',
               'out[0] = rast[0] < 0.2 ? 1 : 0;'
           ) AS rast
    FROM scene
""")
mask.cache()
mask.createOrReplaceTempView("mask")
mask.selectExpr(
    "x",
    "y",
    "RS_BandPixelType(rast, 1) as dtype",
    "RS_Width(rast) as w",
    "RS_Height(rast) as h",
).show()

## 4. Vectorize the mask into a single dissolved flood polygon

`RS_PixelAsPolygons(raster, band)` returns an array of `{geom, val, x, y}` structs — one per pixel. We `explode` that into rows, keep only the wet pixels (`val = 1`), and dissolve them into a single `MultiPolygon` via `ST_Union_Agg`. A small `ST_Buffer` (0.0005°, ~50 m) absorbs SAR speckle along the edge of the flood.

In [ ]:
wet_pixels = sedona.sql("""
    SELECT explode(RS_PixelAsPolygons(rast, 1)) AS pixel
    FROM mask
""").selectExpr("pixel.geom AS geom", "pixel.value AS value").where("value = 1")
wet_pixels.createOrReplaceTempView("wet_pixels")

flood = sedona.sql("""
    SELECT ST_Buffer(ST_Union_Agg(geom), 0.0005) AS geom
    FROM wet_pixels
""")
flood_geom = flood.first()["geom"]
print(
    f"flood polygon: type={flood_geom.geom_type}, "
    f"area={flood_geom.area:.6f} deg², bounds={tuple(round(b, 4) for b in flood_geom.bounds)}"
)

flood.createOrReplaceTempView("flood")

## 4. Vectorize the mask into a single dissolved flood polygon

`RS_PixelAsPolygons(raster, band)` returns an array of `{geom, value, x, y}` structs — one per pixel. We `explode` that into rows, keep only the wet pixels (`value = 1`), and dissolve them into a single `MultiPolygon` via `ST_Union_Agg`. A small `ST_Buffer` (0.0005°, ~50 m) absorbs SAR speckle along the edge of the flood.

In [ ]:
from pyspark.sql import Row

xmin, ymin, xmax, ymax = AOI
step_x = (xmax - xmin) / 4
step_y = (ymax - ymin) / 4
half = 0.00125
types_grid = [
    ["residential", "hospital", "residential", "school"],
    ["residential", "residential", "fire_station", "residential"],
    ["school", "residential", "residential", "hospital"],
    ["residential", "fire_station", "residential", "residential"],
]

building_rows = []
for ix in range(4):
    for iy in range(4):
        cx = xmin + (ix + 0.5) * step_x
        cy = ymin + (iy + 0.5) * step_y
        wkt = (
            f"POLYGON(({cx-half} {cy-half}, {cx+half} {cy-half}, "
            f"{cx+half} {cy+half}, {cx-half} {cy+half}, {cx-half} {cy-half}))"
        )
        building_rows.append(Row(bid=f"B{ix}{iy}", type=types_grid[iy][ix], wkt=wkt))

buildings = sedona.createDataFrame(building_rows).selectExpr(
    "bid", "type", "ST_SetSRID(ST_GeomFromText(wkt), 4326) as geom"
)
buildings.createOrReplaceTempView("buildings")
buildings.show(16)

## 6. Find affected buildings + flag critical infrastructure

`ST_Intersects(building, flood)` is the canonical "affected by" test. We add a `is_critical` flag for any affected building whose type is hospital / school / fire_station — those are the rows an emergency-management dashboard would surface first.

In [ ]:
affected = sedona.sql("""
    SELECT b.bid,
           b.type,
           b.type IN ('hospital', 'school', 'fire_station') AS is_critical,
           b.geom
    FROM buildings b, flood f
    WHERE ST_Intersects(b.geom, f.geom)
    ORDER BY is_critical DESC, b.bid
""")
affected.cache()
affected.createOrReplaceTempView("affected")
affected.select("bid", "type", "is_critical").show()

summary = sedona.sql("""
    SELECT COUNT(*) AS affected_buildings,
           SUM(CAST(is_critical AS INT)) AS affected_critical
    FROM affected
""")
summary.show()

## 7. Persist the affected buildings as GeoParquet 1.1

Auto-populated covering-bbox metadata + projjson CRS (derived from SRID 4326) make the result file ready for downstream tools to filter on bbox without scanning every row. Round-trip read-back confirms the metadata lands.

In [ ]:
import shutil

out_path = f"{WORK}/affected_buildings.parquet"
if os.path.exists(out_path):
    shutil.rmtree(out_path)
affected.coalesce(1).write.format("geoparquet").save(out_path)

sedona.read.format("geoparquet").load(out_path).select(
    "bid", "type", "is_critical"
).show(truncate=False)

## 8. Visualize: backscatter + flood polygon + buildings color-coded

Single matplotlib axes: VV backscatter as the basemap, the dissolved flood polygon overlaid in blue, building footprints filled by status — red for affected critical, orange for affected residential, white for unaffected.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

buildings_pdf = buildings.selectExpr("bid", "type", "ST_AsText(geom) as wkt").toPandas()
affected_ids = set(r["bid"] for r in affected.select("bid").collect())
critical_affected_ids = set(
    r["bid"] for r in affected.where("is_critical").select("bid").collect()
)

flood_wkt = sedona.sql("SELECT ST_AsText(geom) as wkt FROM flood").first()["wkt"]

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
extent = (AOI[0], AOI[2], AOI[1], AOI[3])
ax.imshow(backscatter, vmin=0, vmax=1, cmap="gray", extent=extent)

from shapely import wkt as shapely_wkt

flood_shapely = shapely_wkt.loads(flood_wkt)
for poly in (
    flood_shapely.geoms if hasattr(flood_shapely, "geoms") else [flood_shapely]
):
    xs_p, ys_p = poly.exterior.xy
    ax.fill(
        xs_p, ys_p, facecolor="#268bd2", edgecolor="#073642", linewidth=1, alpha=0.5
    )

for _, row in buildings_pdf.iterrows():
    bbox = row["wkt"].split("((")[1].split("))")[0]
    pts = [tuple(float(x) for x in p.strip().split()) for p in bbox.split(",")]
    xs_b = [p[0] for p in pts]
    ys_b = [p[1] for p in pts]
    if row["bid"] in critical_affected_ids:
        face = "#dc322f"  # red
    elif row["bid"] in affected_ids:
        face = "#cb4b16"  # orange
    else:
        face = "#fdf6e3"  # cream
    ax.fill(xs_b, ys_b, facecolor=face, edgecolor="black", linewidth=0.6)
    ax.text(
        sum(xs_b[:-1]) / 4,
        sum(ys_b[:-1]) / 4,
        f"{row['bid']}\n{row['type'][:4]}",
        ha="center",
        va="center",
        fontsize=6,
    )

ax.set_title(
    f"Flood snapshot: {len(affected_ids)} buildings affected, "
    f"{len(critical_affected_ids)} critical"
)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.legend(
    handles=[
        mpatches.Patch(facecolor="#268bd2", alpha=0.5, label="flood extent"),
        mpatches.Patch(facecolor="#dc322f", label="affected critical"),
        mpatches.Patch(facecolor="#cb4b16", label="affected residential"),
        mpatches.Patch(facecolor="#fdf6e3", edgecolor="black", label="unaffected"),
    ],
    loc="upper right",
    fontsize=8,
)
fig.tight_layout()
fig

## What just happened?

An end-to-end disaster-response pipeline in a handful of SQL passes:

1. Synthesized a SAR-VV-style backscatter raster.
2. Thresholded with single-raster `RS_MapAlgebra` to derive a binary water mask.
3. Vectorized the mask via `RS_PixelAsPolygons` + `explode` + `ST_Union_Agg` and absorbed speckle with `ST_Buffer` — the canonical raster→vector dissolve in Sedona.
4. Built a Spark DataFrame of building footprints labelled with infrastructure type.
5. `ST_Intersects` spatial join surfaced the buildings inside the flood; a `type IN (...)` flag marked the critical ones.
6. Persisted to GeoParquet 1.1 (auto covering-bbox + projjson CRS), read back, plotted everything on a single matplotlib axes.

Replace the synthesis cell in section 2 with `sedona.read.format("raster").load("<sentinel-1 vv asset URL discovered via sedona.read.format('stac')>")` and the rest of the pipeline runs unchanged on real Copernicus data.